# HMS merge

按章节合并 Lecture、按 Exercise 合并练习、按时间顺序合并全部考试；最后一个 cell 单独翻译所有合并后的 PDF。

In [1]:
from pathlib import Path
import re
from collections import defaultdict

HMS_DIR = Path(r"E:\OneDrive - MSFT\.master_data\26ss\Machine Vision2526ws")
LECTURE_DIR = HMS_DIR / "Slides"
# EXERCISE_DIR = HMS_DIR / "Exercise Material"
EXAM_DIR = HMS_DIR / "Old exams"
EXTS = {".ppt", ".pptx", ".pdf"}

def numeric_key(path):
    return tuple(int(x) for x in re.findall(r"\d+", path.stem))

def count_pdf_pages(path):
    from pypdf import PdfReader
    return len(PdfReader(str(path)).pages)

def merge_pdfs(files, output_path):
    from pypdf import PdfWriter
    writer = PdfWriter()
    file_pages = []
    for path in files:
        pages = count_pdf_pages(path)
        file_pages.append((path, pages))
        writer.append(str(path))
    output_path.parent.mkdir(exist_ok=True)
    with output_path.open("wb") as output:
        writer.write(output)
    return file_pages, count_pdf_pages(output_path)

def merge_ppts(files, output_path):
    try:
        import win32com.client as win32
    except ImportError:
        raise ImportError("请先运行：pip install pywin32；该方法需要 Windows + PowerPoint。")
    app = win32.Dispatch("PowerPoint.Application")
    app.Visible = True
    merged = app.Presentations.Add()
    file_pages = []
    try:
        while merged.Slides.Count > 0:
            merged.Slides(1).Delete()
        for path in files:
            src = app.Presentations.Open(str(path.resolve()), ReadOnly=True, WithWindow=False)
            pages = src.Slides.Count
            file_pages.append((path, pages))
            src.Close()
            if pages > 0:
                merged.Slides.InsertFromFile(str(path.resolve()), merged.Slides.Count, 1, pages)
        total_pages = merged.Slides.Count
        merged.SaveAs(str(output_path.resolve()))
    finally:
        merged.Close()
        app.Quit()
    return file_pages, total_pages

def merge_group(files, output_dir, group):
    files = sorted(files, key=numeric_key)
    ppt_files = [p for p in files if p.suffix.lower() in {".ppt", ".pptx"}]
    pdf_files = [p for p in files if p.suffix.lower() == ".pdf"]
    output_dir.mkdir(exist_ok=True)
    if ppt_files:
        pages, total = merge_ppts(ppt_files, output_dir / f"{group}.pptx")
        print(f"{group}.pptx: {len(pages)} files, {total} slides")
    if pdf_files:
        pages, total = merge_pdfs(pdf_files, output_dir / f"{group}.pdf")
        print(f"{group}.pdf: {len(pages)} files, {total} pages")


## 1. Merge lecture material by chapter

In [7]:
# 按章节顺序合并所有 Lecture，并为每个 Chapter 添加一级 PDF 目录。
LECTURE_OUTPUT_DIR = LECTURE_DIR 
from pypdf import PdfReader, PdfWriter

def chapter_file_key(path):
    match = re.match(r"^(\d{2})-", path.stem, re.IGNORECASE)
    return int(match.group(1)) if match else 9999

def chapter_outline_title(path):
    match = re.match(r"^(\d{2})-(.+)$", path.stem, re.IGNORECASE)
    return f"{match.group(1)}: {match.group(2).replace('-', ' ')}"

chapter_files = sorted([
    path for path in LECTURE_OUTPUT_DIR.glob("*.pdf")
    if re.match(r"^\d{2}-.+", path.stem, re.IGNORECASE)
    and not path.name.startswith("trans-")
], key=chapter_file_key)

if not chapter_files:
    raise FileNotFoundError(f"没有找到带名称的 Chapter PDF：{LECTURE_OUTPUT_DIR}")

all_chapters_output = LECTURE_OUTPUT_DIR / "_merged" / "All_Chapters.pdf"
writer = PdfWriter()
total_pages = 0
for path in chapter_files:
    pages = len(PdfReader(str(path)).pages)
    writer.append(str(path), outline_item=chapter_outline_title(path), import_outline=False)
    total_pages += pages
    print(f"{path.name} -> {pages} pages")

with all_chapters_output.open("wb") as output:
    writer.write(output)
print(f"合并完成：{all_chapters_output}")
print(f"Chapter 数量：{len(chapter_files)}，总页数：{total_pages}")


01-Introduction.pdf -> 19 pages
02-Preprocessing.pdf -> 68 pages
03-Edgedetection.pdf -> 31 pages
04-Curvefitting.pdf -> 54 pages
05-Color.pdf -> 23 pages
06-Segmentation.pdf -> 84 pages
07-Optics.pdf -> 70 pages
10-Patternrecognition.pdf -> 112 pages
12-DeepLearningBasics.pdf -> 64 pages
13-DeepLearningTransformers.pdf -> 39 pages
14-DeepLearningGenerative.pdf -> 16 pages
合并完成：E:\OneDrive - MSFT\.master_data\26ss\Machine Vision2526ws\Slides\_merged\All_Chapters.pdf
Chapter 数量：11，总页数：580


## 2. Merge exercise material by exercise number

## 3. Merge all exam preparation material

In [11]:
def exam_sort_key(path):
    text = path.stem.lower()

    # match = re.search(r"(ws|ss)(\d{2,4})(\d{2})?", text)
    match = re.search(r"machinevision_(\d{2})(f2|f|h|p)", text)

    if match:
        # digits = match.group(2)
        # first = int(digits)
        # if len(digits) == 4 and int(digits[2:]) == int(digits[:2]) + 1:
        #     year = 2000 + int(digits[:2])  # WS1516 -> 2015, WS1920 -> 2019
        # else:
        #     year = first if first >= 1900 else 2000 + first
        year = 2000 + int(match.group(1))
        period = match.group(2)
    else:
        # years = [int(x) for x in re.findall(r"(?:19|20)\d{2}", text)]
        # year = min(years, default=9999)
        year = 9999
        period = ""

    # term = 0 if "ss" in text else 1
    term_order = {"f": 0, "f2": 1, "h": 2, "p": 3}
    term = term_order.get(period, 9)

    # solution = 1 if "solution" in text else 0
    solution = 1 if "loesungen" in text else 0

    return year, term, solution, numeric_key(path), text


exam_files = sorted([
    path for path in EXAM_DIR.rglob("*.pdf")
    if path.is_file() and not path.name.startswith("~$") and "_merged" not in path.parts
], key=exam_sort_key)

if not exam_files:
    raise FileNotFoundError(f"没有找到考试 PDF: {EXAM_DIR}")


from pypdf import PdfReader, PdfWriter


exam_output = EXAM_DIR / "_merged" / "MV_Exams.pdf"


def exam_period(path):
    # match = re.search(r"(?:^|_)(WS|SS)(\d{2,4})", path.stem, re.IGNORECASE)
    # return match.group(0).strip("_").upper() if match else path.stem
    match = re.search(r"MachineVision_(\d{2})(F2|F|H|P)", path.stem, re.IGNORECASE)
    return f"{match.group(1)}{match.group(2).upper()}" if match else path.stem


def exam_title(path):
    # kind = "Solution" if "solution" in path.stem.lower() else "Task"
    kind = "Solution" if "loesungen" in path.stem.lower() else "Task"
    return f"{exam_period(path)} - {kind}"


# TASK_HEADING = re.compile(r"^(Task|Problem)\s+(\d+)(?![\d.])\s*:?[ \t]*(.*)$", re.IGNORECASE)
TASK_HEADING = re.compile(r"^(Task|Problem|Question)\s+(\d+)(?![\d.])\s*:?[ \t]*(.*)$", re.IGNORECASE)


def extract_task_bookmarks(path):
    reader = PdfReader(str(path))
    bookmarks = []
    seen = set()

    for page_index, page in enumerate(reader.pages):
        lines = [" ".join(line.split()) for line in (page.extract_text() or "").splitlines() if line.strip()]
        page_entries = []

        for line in lines:
            # 部分 PDF 会把 Task 抽成 T ask。
            line = re.sub(r"\bT\s+ask\b", "Task", line, flags=re.IGNORECASE)
            line = re.sub(r"\bP\s+roblem\b", "Problem", line, flags=re.IGNORECASE)

            # 新增：部分 PDF 可能会把 Question 抽成 Q uestion。
            line = re.sub(r"\bQ\s+uestion\b", "Question", line, flags=re.IGNORECASE)

            match = TASK_HEADING.match(line)

            if match and match.group(2) not in seen:
                page_entries.append((match.group(2), line))

        # 第一页通常是总目录，真正的 Task 从后面的页面开始。
        # if page_index == 0 and len({number.split(".")[0] for number, _ in page_entries}) > 1:
        #     continue

        for number, title in page_entries:
            if number not in seen:
                seen.add(number)
                title = re.sub(r"\s+\d+(?:\s+\d+)*\s*$", "", title)
                bookmarks.append((number, title, page_index))

    return bookmarks


writer = PdfWriter()
total = 0

for path in exam_files:
    count = count_pdf_pages(path)
    page_start = total
    writer.append(str(path), import_outline=False)
    exam_node = writer.add_outline_item(exam_title(path), page_number=page_start)
    task_bookmarks = extract_task_bookmarks(path)

    for number, title, page_index in task_bookmarks:
        absolute_page = page_start + page_index
        writer.add_outline_item(title, page_number=absolute_page, parent=exam_node)

    total += count
    print(f"{path.name} -> {count} pages, {len(task_bookmarks)} task bookmarks")


exam_output.parent.mkdir(exist_ok=True)

with exam_output.open("wb") as output:
    writer.write(output)

print(f"MV_Exams.pdf: {len(exam_files)} files, {total} pages")

MachineVision_16F.pdf -> 4 pages, 9 task bookmarks
MachineVision_16F_Loesungen.pdf -> 7 pages, 9 task bookmarks
MachineVision_16H.pdf -> 5 pages, 9 task bookmarks
MachineVision_16H_Loesungen.pdf -> 8 pages, 9 task bookmarks
MachineVision_16P.pdf -> 4 pages, 8 task bookmarks
MachineVision_16P_Loesungen.pdf -> 7 pages, 8 task bookmarks
MachineVision_17F.pdf -> 4 pages, 9 task bookmarks
MachineVision_17F_Loesungen.pdf -> 11 pages, 9 task bookmarks
MachineVision_17H.pdf -> 5 pages, 9 task bookmarks
MachineVision_17H_Loesungen.pdf -> 8 pages, 9 task bookmarks
MachineVision_18F.pdf -> 4 pages, 9 task bookmarks
MachineVision_18F_Loesungen.pdf -> 8 pages, 9 task bookmarks
MachineVision_18H.pdf -> 5 pages, 8 task bookmarks
MachineVision_18H_Loesungen.pdf -> 7 pages, 8 task bookmarks
MachineVision_19F.pdf -> 19 pages, 9 task bookmarks
MachineVision_19F_Loesungen.pdf -> 13 pages, 9 task bookmarks
MachineVision_19H.pdf -> 19 pages, 9 task bookmarks
MachineVision_19H_Loesungen.pdf -> 12 pages, 9 ta

## 4. Translate all merged PDFs (run separately after merging)

In [ ]:
from pathlib import Path
import subprocess
import tempfile
import shutil

OVERWRITE = False
pdf2zh_next_cmd = shutil.which("pdf2zh_next")
if pdf2zh_next_cmd is None:
    raise RuntimeError("没有找到 pdf2zh_next。请先运行：pip install pdf2zh-next")

FOLDERS = [LECTURE_DIR / "_merged", EXAM_DIR / "_merged"]
pdf_files = sorted([
    p for folder in FOLDERS for p in folder.glob("*.pdf")
    if p.is_file()
    and not p.name.startswith("trans-")
    and not p.name.startswith("~$")
])
print("=" * 80)
print(f"待翻译文件夹：{FOLDERS}")
print(f"待翻译 PDF 数量：{len(pdf_files)}")
print("=" * 80)

for pdf_path in pdf_files:
    out_path = pdf_path.with_name("trans-" + pdf_path.name)
    if out_path.exists() and not OVERWRITE:
        print(f"跳过，已存在：{out_path.name}")
        continue
    print("\n" + "-" * 80)
    print(f"开始翻译：{pdf_path.name}")
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        tmp_input = tmpdir / pdf_path.name
        shutil.copy2(pdf_path, tmp_input)
        cmd = [
            pdf2zh_next_cmd,
            str(tmp_input),
            "--lang-in", "en",
            "--lang-out", "zh-CN",
            "--no-mono",
            # 翻译论文要把这俩注释掉，因为 PDF 不能断行，不然翻译不连续；PPT 短句子要开
            "--split-short-lines",
            "--short-line-split-factor", "2.0",
            # "--disable-rich-text-translate",
            "--ignore-cache",
            "--watermark-output-mode", "no_watermark",
            # "--enhance-compatibility",
            # "--use-alternating-pages-dual",
            
            # 提速
            "--qps", "20",
            "--pool-max-workers", "100",

            # # 如果不需要自动术语提取，也建议关闭
            # "--no-auto-extract-glossary",
        ]
        result = subprocess.run(
            cmd,
            cwd=tmpdir,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT
        )
        print(result.stdout[-2000:])
        if result.returncode != 0:
            print(f"失败：{pdf_path.name}")
            continue
        candidates = list(tmpdir.glob(f"{pdf_path.stem}*dual*.pdf"))
        if not candidates:
            candidates = list(tmpdir.glob("*.pdf"))
            candidates = [p for p in candidates if p.name != pdf_path.name]
        if not candidates:
            print(f"没有找到翻译输出文件：{pdf_path.name}")
            continue
        if out_path.exists() and OVERWRITE:
            out_path.unlink()
        translated_pdf = candidates[0]
        shutil.move(str(translated_pdf), str(out_path))
        print(f"完成：{out_path.name}")
print("\n" + "=" * 80)
print("全部处理完成")
print("=" * 80)


待翻译文件夹：[WindowsPath('E:/OneDrive - MSFT/.master_data/25-26ws/hms/Lecture Material/_merged'), WindowsPath('E:/OneDrive - MSFT/.master_data/25-26ws/hms/Exercise Material/_merged'), WindowsPath('E:/OneDrive - MSFT/.master_data/25-26ws/hms/Exam Preparation Material/_merged')]
待翻译 PDF 数量：24

--------------------------------------------------------------------------------
开始翻译：HMS_Exams.pdf
DetectScannedFile (1/1)                                ----- 674/… 0:00… 0:00:…
Parse Page Layout (1/1)                                ----- 1348… 0:08… 0:00:…
Parse Paragraphs (1/1)                                 ----- 674/… 0:00… 0:00:…
Parse Formulas and Styles (1/1)                        ----- 674/… 0:00… 0:00:…
Automatic Term Extraction (1/1)                        ----- 2350… 0:02… 0:00:…
Translate Paragraphs (1/1)                             ----- 2350… 0:05… 0:00:…
Typesetting (1/1)                                      ----- 1348… 0:00… 0:00:…
Add Fonts (1/1)                                     

In [1]:
# 将原文和翻译版总 Exam 分别按大题主题归类：一个主题一个 PDF。
from collections import defaultdict
from pathlib import Path
from pypdf import PdfReader, PdfWriter

EXAM_MERGED_DIR = Path(r"E:\OneDrive - MSFT\.master_data\25-26ws\hms\Exam Preparation Material\_merged")
EXAM_SOURCE_FILES = [
    EXAM_MERGED_DIR / "HMS_Exams.pdf",
    EXAM_MERGED_DIR / "trans-HMS_Exams.pdf",
]

# 顺序就是输出文件的优先级顺序。
TASK_CATEGORIES = [
    ("01_9-valued_Logic", ("9-valued", "ieee-1164")),
    ("02_Fault_Simulation", ("fault simulation",)),
    ("03_Design_of_a_VHDL_Model", ("design of a vhdl model",)),
    ("04_SystemC", ("systemc",)),
    ("05_VHDL-AMS", ("vhdl-ams", "vhdl ams")),
    ("06_Synthesis", ("synthesis", "ordered binary decision diagram")),
    ("07_Circuit_Simulation", ("circuit simulation",)),
    ("08_Timing_Delta_Cycles", ("timing", "delta cycles", "events")),
]

def classify_task(title):
    text = title.lower()
    for category, keywords in TASK_CATEGORIES:
        if any(keyword in text for keyword in keywords):
            return category
    return None

def collect_task_segments(reader):
    # 总 Exam 的一级目录是年份/Task 或 Solution，下一层是各大题。
    exams = []
    outline = reader.outline
    index = 0
    while index < len(outline):
        exam_item = outline[index]
        children = outline[index + 1] if index + 1 < len(outline) and isinstance(outline[index + 1], list) else []
        exams.append((exam_item, children))
        index += 2 if children else 1

    segments = defaultdict(list)
    unclassified = set()
    exam_starts = [reader.get_destination_page_number(exam_item) for exam_item, _ in exams]
    exam_starts.append(len(reader.pages))

    for exam_index, (exam_item, tasks) in enumerate(exams):
        exam_end = exam_starts[exam_index + 1]
        for task_index, task_item in enumerate(tasks):
            start_page = reader.get_destination_page_number(task_item)
            end_page = (
                reader.get_destination_page_number(tasks[task_index + 1])
                if task_index + 1 < len(tasks) else exam_end
            )
            category = classify_task(task_item.title)
            if category:
                bookmark = f"{exam_item.title}: {task_item.title}"
                segments[category].append((start_page, end_page, bookmark))
            else:
                unclassified.add(task_item.title)
    return segments, sorted(unclassified)

for source_path in EXAM_SOURCE_FILES:
    if not source_path.exists():
        print(f"跳过，不存在：{source_path}")
        continue

    reader = PdfReader(str(source_path))
    segments, unclassified = collect_task_segments(reader)
    output_dir = EXAM_MERGED_DIR / "_by_task" / source_path.stem
    output_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"输入文件：{source_path.name}")
    for category, _ in TASK_CATEGORIES:
        task_segments = segments.get(category, [])
        if not task_segments:
            print(f"没有找到：{category}")
            continue

        writer = PdfWriter()
        for start_page, end_page, bookmark in task_segments:
            output_start = len(writer.pages)
            for page_index in range(start_page, end_page):
                writer.add_page(reader.pages[page_index])
            writer.add_outline_item(bookmark, page_number=output_start)

        output_path = output_dir / f"{category}.pdf"
        with output_path.open("wb") as output:
            writer.write(output)
        print(f"{output_path.name}: {len(task_segments)} sections, {len(writer.pages)} pages")

    if unclassified:
        print(f"未归类并跳过：{unclassified}")



输入文件：HMS_Exams.pdf
01_9-valued_Logic.pdf: 21 sections, 69 pages
02_Fault_Simulation.pdf: 21 sections, 49 pages
03_Design_of_a_VHDL_Model.pdf: 21 sections, 102 pages
04_SystemC.pdf: 11 sections, 47 pages
05_VHDL-AMS.pdf: 21 sections, 89 pages
06_Synthesis.pdf: 21 sections, 104 pages
07_Circuit_Simulation.pdf: 21 sections, 56 pages
08_Timing_Delta_Cycles.pdf: 31 sections, 101 pages
未归类并跳过：['Problem', 'Task 1: Multiple Choice']

输入文件：trans-HMS_Exams.pdf
01_9-valued_Logic.pdf: 21 sections, 69 pages
02_Fault_Simulation.pdf: 21 sections, 49 pages
03_Design_of_a_VHDL_Model.pdf: 21 sections, 102 pages
04_SystemC.pdf: 11 sections, 47 pages
05_VHDL-AMS.pdf: 21 sections, 89 pages
06_Synthesis.pdf: 21 sections, 104 pages
07_Circuit_Simulation.pdf: 21 sections, 56 pages
08_Timing_Delta_Cycles.pdf: 31 sections, 101 pages
未归类并跳过：['Problem', 'Task 1: Multiple Choice']
